# Framing hits — overview

Load **`outputs/framing_gpt_results.csv`**, then category × outlet summaries and hit-text exploration.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
for _p in (_here, _here / "03_Framing", _here.parent / "03_Framing"):
    if (_p / "framing_paths.py").is_file():
        sys.path.insert(0, str(_p))
        break

import pandas as pd
from framing_paths import resolve_framing_paths

THESIS_ROOT, FRAMING_ROOT, OUT_DIR = resolve_framing_paths()
FRAMING_CSV = OUT_DIR / "framing_gpt_results.csv"
CORPUS_CSV = THESIS_ROOT / "01_EDAperOutlet" / "outputs" / "df_combined.csv"

df = pd.read_csv(FRAMING_CSV)
csv_path = FRAMING_CSV

print(f"Loaded {len(df):,} rows from {FRAMING_CSV}")
print(f"Thesis root: {THESIS_ROOT}")

from IPython.display import display


### Data Exploration - Framing Results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df["source"] = df["source"].fillna("unknown").astype(str).str.strip()
df["category"] = df["category"].fillna("missing").astype(str).str.strip()

category_order = [
    "POSITIONS-/PARTEILICHKEITS-BIAS",
    "VERZERRUNG/MANIPULATION",
    "DISINFORMATION/FALSCHDARSTELLUNG",
    "VERSAGEN/INKOMPETENZ",
    "NEUTRAL",
    "IRRELEVANT",
]

source_order = df["source"].value_counts().index.tolist()

print(f"Rows: {len(df):,}")
display(df.head())


In [ ]:
overall_category_counts = (
    df["category"]
    .value_counts()
    .reindex(category_order, fill_value=0)
    .rename_axis("category")
    .reset_index(name="n")
)

overall_category_counts["pct"] = overall_category_counts["n"] / overall_category_counts["n"].sum() * 100
display(overall_category_counts.round(2))


In [ ]:
category_by_source_counts = pd.crosstab(
    df["source"],
    df["category"],
).reindex(index=source_order, columns=category_order, fill_value=0)

display(category_by_source_counts)
category_by_source_pct = (
    category_by_source_counts
    .div(category_by_source_counts.sum(axis=1), axis=0)
    .mul(100)
    .round(2)
)

display(category_by_source_pct)


In [ ]:
plt.figure(figsize=(12, 7))
category_by_source_counts.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 7),
    colormap="tab20"
)
plt.title("Category Distribution by Source")
plt.xlabel("Source")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Uses THESIS_ROOT, FRAMING_CSV, CORPUS_CSV from the first code cell

df = pd.read_csv(FRAMING_CSV).copy()
corpus_df = pd.read_csv(CORPUS_CSV, usecols=["row_id"]).copy()

# Clean
df["source"] = df["source"].fillna("unknown").astype(str).str.strip()
df["category"] = df["category"].fillna("missing").astype(str).str.strip()
df["hit_text"] = df["hit_text"].fillna("").astype(str).str.strip()

NON_NEUTRAL_EXCLUDE = {"NEUTRAL", "IRRELEVANT"}
df["is_non_neutral"] = ~df["category"].isin(NON_NEUTRAL_EXCLUDE)

# KPIs
total_corpus_articles = corpus_df["row_id"].nunique()
articles_mentioning_mainstream_media = df["row_id"].nunique()
articles_with_bias = df.loc[df["is_non_neutral"], "row_id"].nunique()

pct_articles_with_mentions = (
    articles_mentioning_mainstream_media / total_corpus_articles * 100
    if total_corpus_articles > 0 else 0
)

pct_articles_with_bias_given_mentions = (
    articles_with_bias / articles_mentioning_mainstream_media * 100
    if articles_mentioning_mainstream_media > 0 else 0
)

total_entity_hits = len(df)
non_neutral_frames = int(df["is_non_neutral"].sum())

# Source-level rates
source_stats = (
    df.groupby("source")
    .agg(
        total_hits=("category", "size"),
        non_neutral_hits=("is_non_neutral", "sum"),
    )
    .reset_index()
)

source_stats["non_neutral_rate"] = (
    source_stats["non_neutral_hits"] / source_stats["total_hits"] * 100
)

source_stats = source_stats.sort_values("non_neutral_rate", ascending=True).reset_index(drop=True)

# Print KPIs as plain text
print(f"Total corpus articles: {total_corpus_articles:,}")
print(
    f"Articles mentioning mainstream media: {articles_mentioning_mainstream_media:,} "
    f"({pct_articles_with_mentions:.1f}% of corpus articles)"
)
print(
    f"Articles with bias: {articles_with_bias:,} "
    f"({pct_articles_with_bias_given_mentions:.1f}% of articles mentioning mainstream media)"
)
print(f"Total entity hits: {total_entity_hits:,}")
print(f"Non-neutral frames: {non_neutral_frames:,}")

# Plot
plt.figure(figsize=(10, 6))
plt.barh(
    source_stats["source"],
    source_stats["non_neutral_rate"],
    color="steelblue"
)

for _, row in source_stats.iterrows():
    plt.text(
        row["non_neutral_rate"] + 0.5,
        row["source"],
        f"{row['non_neutral_rate']:.1f}%",
        va="center"
    )

plt.xlabel("Non-neutral frame rate (%)")
plt.ylabel("Source")
plt.title("Non-neutral frame rate across outlets")
plt.xlim(0, max(65, source_stats["non_neutral_rate"].max() + 5))
plt.tight_layout()
plt.show()


In [ ]:
# Original categories in the data
bias_categories = [
    "POSITIONS-/PARTEILICHKEITS-BIAS",
    "VERZERRUNG/MANIPULATION",
    "DISINFORMATION/FALSCHDARSTELLUNG",
    "VERSAGEN/INKOMPETENZ",
]

# English labels only for display
category_labels = {
    "POSITIONS-/PARTEILICHKEITS-BIAS": "Agenda /\nAllegiance Bias",
    "VERZERRUNG/MANIPULATION": "Distortion /\nManipulation Bias",
    "DISINFORMATION/FALSCHDARSTELLUNG": "Disinformation /\nFalsehood Bias",
    "VERSAGEN/INKOMPETENZ": "Failure /\nIncompetence Bias",
}

plot_df = df[
    (df["source"] != "Tagesschau") &
    (df["category"].isin(bias_categories))
].copy()

source_order = [
    s for s in [
        "Antispiegel", "Compact", "Deutschlandkurier",
        "Nius", "RT_de", "Tichys_Einblick"
    ]
    if s in plot_df["source"].unique()
]

bias_counts = pd.crosstab(plot_df["source"], plot_df["category"]).reindex(
    index=source_order,
    columns=bias_categories,
    fill_value=0,
)

bias_pct = (
    bias_counts
    .div(bias_counts.sum(axis=1), axis=0)
    .mul(100)
    .round(1)
)

# Rename columns only for plotting/display
bias_pct_plot = bias_pct.rename(columns=category_labels)

display(bias_counts)
display(bias_pct_plot)

plt.figure(figsize=(10, 5))
sns.heatmap(bias_pct_plot, annot=True, fmt=".1f", cmap="Blues")
plt.title("Bias Focus by Alternative Outlet (%)")
plt.xlabel("Bias category")
plt.ylabel("Outlet")
plt.tight_layout()
plt.show()

In [ ]:
# English labels only for display
category_labels = {
    "POSITIONS-/PARTEILICHKEITS-BIAS": "Agenda /\nAllegiance Bias",
    "VERZERRUNG/MANIPULATION": "Distortion /\nManipulation Bias",
    "DISINFORMATION/FALSCHDARSTELLUNG": "Disinformation /\nFalsehood Bias",
    "VERSAGEN/INKOMPETENZ": "Failure /\nIncompetence Bias",
}

bias_pct_plot = bias_pct.rename(columns=category_labels)

bias_pct_plot.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6),
    colormap="tab20c"
)

plt.title("Bias Focus by Alternative Outlet (%)")
plt.xlabel("Outlet")
plt.ylabel("Percent of non-neutral bias-coded rows")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Bias category", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12, 7))
sns.heatmap(category_by_source_pct, annot=True, fmt=".1f", cmap="Blues")
plt.title("Category Shares by Source (%)")
plt.xlabel("Category")
plt.ylabel("Source")
plt.tight_layout()
plt.show()


In [ ]:
non_neutral_df = df[
    (df["category"] != "NEUTRAL") &
    (df["source"] != "Tagesschau")
].copy()

non_neutral_source_order = [s for s in source_order if s != "Tagesschau"]

non_neutral_counts = pd.crosstab(
    non_neutral_df["source"],
    non_neutral_df["category"],
).reindex(
    index=non_neutral_source_order,
    columns=[c for c in category_order if c != "NEUTRAL"],
    fill_value=0
)

non_neutral_pct = (
    non_neutral_counts
    .div(non_neutral_counts.sum(axis=1), axis=0)
    .mul(100)
    .round(2)
)

display(non_neutral_counts)
display(non_neutral_pct)

plt.figure(figsize=(12, 7))
sns.heatmap(non_neutral_pct, annot=True, fmt=".1f", cmap="Oranges")
plt.title("Non-Neutral Category Shares by Source (%) Without Tagesschau")
plt.xlabel("Category")
plt.ylabel("Source")
plt.tight_layout()
plt.show()


### Hit Text Exploration

Overall Hit Text Distribution

In [ ]:
hit_text_counts = (
    df["hit_text"]
    .fillna("<<MISSING>>")
    .value_counts(dropna=False)
    .rename_axis("hit_text")
    .reset_index(name="frequency")
)

hit_text_counts["pct_overall"] = (hit_text_counts["frequency"] / len(df) * 100).round(2)

print(f"Total rows: {len(df):,}")
print(f"Unique hit_text values (including missing): {hit_text_counts.shape[0]:,}")
hit_text_counts.head(50)

Hit Text Distirbution: Biased mentions

In [ ]:
non_neutral_irrelevant_df = df[~df["category"].isin(["NEUTRAL", "IRRELEVANT"])].copy()

hit_text_counts = (
    non_neutral_irrelevant_df["hit_text"]
    .fillna("<<MISSING>>")
    .value_counts(dropna=False)
    .rename_axis("hit_text")
    .reset_index(name="frequency")
)

hit_text_counts["pct_overall"] = (hit_text_counts["frequency"] / len(non_neutral_irrelevant_df) * 100).round(2)

print(f"Total rows: {len(non_neutral_irrelevant_df):,}")
print(f"Unique hit_text values (including missing): {hit_text_counts.shape[0]:,}")
hit_text_counts.head(50)

In [ ]:
# Save current hit_text_counts to CSV
out_path = OUT_DIR / "hit_text_counts.csv"
hit_text_counts.to_csv(out_path, index=False, encoding="utf-8-sig")

print(f"Saved: {out_path}")


In [ ]:
# Comparative outlet framing dashboard (cell 14)

focus_categories = [
    "POSITIONS-/PARTEILICHKEITS-BIAS",
    "VERZERRUNG/MANIPULATION",
    "DISINFORMATION/FALSCHDARSTELLUNG",
    "VERSAGEN/INKOMPETENZ",
]

# 3) Difference from corpus baseline in percentage points (all outlets)
baseline_pct = overall_category_counts.set_index("category")["pct"]
delta_pp = (
    category_by_source_pct.loc[source_order, focus_categories]
    .sub(baseline_pct[focus_categories], axis=1)
    .round(2)
)

fig, axes = plt.subplots(
    1, 3, figsize=(24, 7),
    gridspec_kw={"width_ratios": [1.0, 1.6, 1.6]}
)

# Panel C: Over/under-indexing vs overall baseline
sns.heatmap(
    delta_pp,
    annot=True, fmt=".1f", center=0, cmap="RdBu_r",
    linewidths=0.4, linecolor="white",
    cbar_kws={"label": "Δ percentage points vs corpus baseline"},
    ax=axes[2]
)
axes[2].set_title("Category Lift by Outlet (vs Overall)")
axes[2].set_xlabel("Category")
axes[2].set_ylabel("")

plt.suptitle("Framing Differences Across Outlets", y=1.02, fontsize=16)
plt.tight_layout()
for ax in axes[:2]:
    fig.delaxes(ax)

fig.set_size_inches(12, 7)
axes[2].set_position([0.08, 0.15, 0.78, 0.75])

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")

# 0. Data already loaded in the first cell (run cells in order)
df = df.copy()  # fresh copy for entity exploration

# Basic cleanup
df["source"] = df["source"].fillna("unknown").astype(str).str.strip()
df["category"] = df["category"].fillna("missing").astype(str).str.strip()
df["hit_text"] = df["hit_text"].fillna("").astype(str).str.strip()

# Use the existing context id if present; otherwise build a stable fallback
if "context_idx" in df.columns:
    df["context_key"] = df["context_idx"].astype(str)
else:
    df["context_key"] = (
        df["row_id"].astype(str) + "||" + df["context_window"].fillna("").astype(str)
    )

# 1. Explode hit_text into one row per entity per context window
# hit_text looks like "ARD | Tagesschau | Spiegel"
df_exp = df.copy()
df_exp["entity"] = df_exp["hit_text"].str.split(r"\s*\|\s*")
df_exp = df_exp.explode("entity").reset_index(drop=True)
df_exp["entity"] = df_exp["entity"].fillna("").astype(str).str.strip()
df_exp = df_exp[df_exp["entity"] != ""].copy()

# 2. Constants
ACCUSATION_CATS = [
    "POSITIONS-/PARTEILICHKEITS-BIAS",
    "VERZERRUNG/MANIPULATION",
    "DISINFORMATION/FALSCHDARSTELLUNG",
    "VERSAGEN/INKOMPETENZ",
]
ALL_CATS = ACCUSATION_CATS + ["NEUTRAL", "IRRELEVANT"]

SHORT_LABELS = {
    "POSITIONS-/PARTEILICHKEITS-BIAS": "Parteilichkeit",
    "VERZERRUNG/MANIPULATION": "Verzerrung",
    "DISINFORMATION/FALSCHDARSTELLUNG": "Disinformation",
    "VERSAGEN/INKOMPETENZ": "Versagen",
}

OUTLET_ORDER = [
    s for s in [
        "Antispiegel", "Compact", "Deutschlandkurier",
        "Nius", "RT_de", "Tagesschau", "Tichys_Einblick"
    ]
    if s in df_exp["source"].unique()
]

# 3. Entity frequency table
entity_totals = (
    df_exp.groupby("entity")["context_key"]
    .nunique()
    .sort_values(ascending=False)
    .rename("n_contexts")
    .reset_index()
)

display(entity_totals.head(30))

# 4. Filter: accusation rows only, min frequency threshold
MIN_CONTEXTS = 5

df_acc = df_exp[df_exp["category"].isin(ACCUSATION_CATS)].copy()

# Deduplicate: same context window can only contribute once per entity/category
df_acc_dedup = df_acc.drop_duplicates(subset=["context_key", "entity", "category"])

top_entities = (
    df_acc_dedup.groupby("entity")["context_key"]
    .nunique()
    .loc[lambda s: s >= MIN_CONTEXTS]
    .sort_values(ascending=False)
    .head(15)
    .index.tolist()
)

print("Top entities used for analysis:")
print(top_entities)

# 5. Build entity × category matrix
matrix = (
    df_acc_dedup[df_acc_dedup["entity"].isin(top_entities)]
    .groupby(["entity", "category"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=ACCUSATION_CATS, fill_value=0)
)

matrix = matrix.loc[matrix.sum(axis=1).sort_values(ascending=False).index]

matrix_pct = matrix.div(matrix.sum(axis=1), axis=0) * 100
matrix_pct.columns = [SHORT_LABELS[c] for c in matrix_pct.columns]

# 6. Heatmap
annot_labels = matrix_pct.copy().astype(str)
for col_short, col_full in zip(matrix_pct.columns, ACCUSATION_CATS):
    annot_labels[col_short] = (
        matrix_pct[col_short].map("{:.0f}%".format)
        + "\n("
        + matrix[col_full].astype(str)
        + ")"
    )

fig, ax = plt.subplots(figsize=(10, max(4, len(matrix_pct) * 0.55)))
sns.heatmap(
    matrix_pct,
    annot=annot_labels,
    fmt="",
    cmap="YlOrRd",
    linewidths=0.4,
    linecolor="white",
    vmin=0, vmax=100,
    cbar_kws={"label": "% of accusation contexts", "shrink": 0.7},
    ax=ax,
)
ax.set_title(
    "Accusation Type Profile by Target Entity\n"
    "Row-normalised; n = distinct context windows with accusation label",
    fontsize=11,
    pad=12,
)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="x", labelsize=10, rotation=20)
ax.tick_params(axis="y", labelsize=10, rotation=0)
plt.tight_layout()
plt.show()

# 7. Neutral / irrelevant rate per entity
all_dedup = df_exp.drop_duplicates(subset=["context_key", "entity", "category"])

neutral_rate = (
    all_dedup[all_dedup["entity"].isin(top_entities)]
    .groupby("entity")["category"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
    .rename("pct")
    .reset_index()
    .pivot(index="entity", columns="category", values="pct")
    .fillna(0)
)

neutral_rate = neutral_rate.reindex(
    columns=["NEUTRAL", "IRRELEVANT"] + ACCUSATION_CATS,
    fill_value=0
).sort_values("NEUTRAL", ascending=False)

print("Neutral / irrelevant rate per entity (% of all contexts):")
display(neutral_rate)

# 8. Outlet × entity accusation rate
outlet_entity = (
    df_acc_dedup[df_acc_dedup["entity"].isin(top_entities)]
    .groupby(["source", "entity"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=OUTLET_ORDER, fill_value=0)
)

outlet_entity_pct = outlet_entity.div(outlet_entity.sum(axis=1), axis=0) * 100
outlet_entity_pct = outlet_entity_pct.fillna(0)

fig2, ax2 = plt.subplots(figsize=(12, 5))
sns.heatmap(
    outlet_entity_pct,
    annot=True,
    fmt=".0f",
    cmap="Blues",
    linewidths=0.3,
    linecolor="white",
    cbar_kws={"label": "% of outlet's accusation contexts", "shrink": 0.7},
    ax=ax2,
)
ax2.set_title(
    "Which Entities Does Each Outlet Accuse?\n"
    "(% of outlet's accusation-coded contexts mentioning entity)",
    fontsize=11,
)
ax2.tick_params(axis="x", labelsize=9, rotation=30)
ax2.tick_params(axis="y", labelsize=9, rotation=0)
plt.tight_layout()
plt.show()


# Grouping Hit Text

In [ ]:
focus_df = df[~df["category"].isin(["NEUTRAL", "IRRELEVANT"])].copy()

# 3. Explode hit_text into one row per entity
focus_df["entity"] = focus_df["hit_text"].fillna("").astype(str).str.split(r"\s*\|\s*")
focus_df = focus_df.explode("entity").reset_index(drop=True)
focus_df["entity"] = focus_df["entity"].fillna("").astype(str).str.strip()
focus_df = focus_df[focus_df["entity"] != ""].copy()

# 4. Canonicalize obvious aliases
canonical_map = {
    "FAZ": "Frankfurter Allgemeine Zeitung",
    "Frankfurter Allgemeine Zeitung": "Frankfurter Allgemeine Zeitung",
    "SZ": "Süddeutsche Zeitung",
    "Süddeutsche Zeitung": "Süddeutsche Zeitung",
    "Bild-Zeitung": "Bild",
    "Bild": "Bild",
    "Das Erste": "ARD",
    "ARD": "ARD",
    "Tagesschau": "Tagesschau",
    "ÖRR": "ÖRR",
}
focus_df["entity_canonical"] = focus_df["entity"].replace(canonical_map)

In [ ]:
def assign_entity_group(entity):
    if entity in {
        "ARD", "ZDF", "Tagesschau", "Deutschlandfunk",
        "NDR", "NDR Info", "WDR", "SWR", "MDR", "rbb", "BR", "ÖRR"
    }:
        return "ÖRR outlets"

    if entity in {"Markus Lanz", "Jan Böhmermann", "Caren Miosga"}:
        return "ÖRR personalities"

    if entity in {
        "Spiegel", "Bild", "Frankfurter Allgemeine Zeitung", "Süddeutsche Zeitung",
        "taz", "Tagesspiegel", "Berliner Zeitung", "Handelsblatt",
        "Stern", "Rheinische Post"
    }:
        return "Legacy print / magazines"

    if entity == "Correctiv":
        return "Fact-checking / Correctiv"

    if entity in {"Reuters", "dpa", "Politico", "RND", "Euronews"}:
        return "Aggregators / agencies"

    if entity in {"The European", "RTL"}:
        return "Private / digital outlets"

    if entity in {
        "Lügenpresse", "Systemmedien", "Staatsmedien", "Staatsfunk", "Staatssender",
        "Mainstreammedien", "Mainstreampresse", "Gleichschaltung",
        "Haltungsmedien", "Haltungsjournalisten", "Haltungsjournalismus",
        "Altmedien", "Gesternmedien", "Regierungsmedien", "Linkspresse",
        "Medienpropaganda", "Propagandasender", "Propagandamaschine",
        "Propagandamedien"
    }: return "Pejorative collective terms"
        
    if entity in {
    "Qualitätsmedien",
    "Qualitätspresse",
    "Qualitätsjournalismus",
    "Qualitätsjournalisten",
    }: return "Qualitäts collective terms"

    if entity in {"Alternativmedien"}:
        return "Alternative media labels"

    return "Other / inspect"


In [ ]:
focus_df["entity_group"] = focus_df["entity_canonical"].apply(assign_entity_group)

# 6. Stable context key and deduplication
if "context_idx" in focus_df.columns:
    focus_df["context_key"] = focus_df["row_id"].astype(str) + "||" + focus_df["context_idx"].astype(str)
else:
    focus_df["context_key"] = (
        focus_df["row_id"].astype(str) + "||" + focus_df["context_window"].fillna("").astype(str)
    )

# One outlet-context-entity_group combination should count once
group_df = focus_df.drop_duplicates(subset=["source", "context_key", "entity_group"]).copy()

# 7. Order
source_order = [
    s for s in [
        "Antispiegel", "Compact", "Deutschlandkurier",
        "Nius", "RT_de", "Tagesschau", "Tichys_Einblick"
    ]
    if s in group_df["source"].unique()
]

group_order = [
    "ÖRR outlets",
    "ÖRR personalities",
    "Legacy print / magazines",
    "Fact-checking / Correctiv",
    "Aggregators / agencies",
    "Private / digital outlets",
    "Pejorative collective terms",
    "Qualitäts collective terms",
    "Alternative media labels",
    "Other / inspect",
]

group_order = [g for g in group_order if g in group_df["entity_group"].unique()]

# 8. Counts and percentages
group_counts = pd.crosstab(group_df["source"], group_df["entity_group"]).reindex(
    index=source_order,
    columns=group_order,
    fill_value=0,
)

group_pct = (
    group_counts
    .div(group_counts.sum(axis=1), axis=0)
    .mul(100)
    .round(2)
)

display(group_counts)
display(group_pct)

# 9. Heatmap of within-outlet percentages
plt.figure(figsize=(13, 6))
sns.heatmap(group_pct, annot=True, fmt=".1f", cmap="Blues")
plt.title("Entity Group Distribution by Outlet (%)")
plt.xlabel("Entity group")
plt.ylabel("Source")
plt.tight_layout()
plt.show()

# 10. Difference from overall baseline in percentage points
overall_group_pct = (
    group_df["entity_group"]
    .value_counts(normalize=True)
    .reindex(group_order, fill_value=0)
    .mul(100)
)

delta_pp = group_pct.sub(overall_group_pct, axis=1).round(2)

plt.figure(figsize=(13, 6))
sns.heatmap(delta_pp, annot=True, fmt=".1f", cmap="RdBu_r", center=0)
plt.title("Over-/Under-Indexing of Entity Groups by Outlet\n(percentage points vs overall baseline)")
plt.xlabel("Entity group")
plt.ylabel("Source")
plt.tight_layout()
plt.show()

In [ ]:
group_df_no_tagesschau = group_df[group_df["source"] != "Tagesschau"].copy()

source_order_no_tagesschau = [s for s in source_order if s != "Tagesschau"]

group_counts_no_tagesschau = pd.crosstab(
    group_df_no_tagesschau["source"],
    group_df_no_tagesschau["entity_group"]
).reindex(
    index=source_order_no_tagesschau,
    columns=group_order,
    fill_value=0,
)

group_pct_no_tagesschau = (
    group_counts_no_tagesschau
    .div(group_counts_no_tagesschau.sum(axis=1), axis=0)
    .mul(100)
    .round(2)
)

display(group_counts_no_tagesschau)
display(group_pct_no_tagesschau)

plt.figure(figsize=(13, 6))
sns.heatmap(group_pct_no_tagesschau, annot=True, fmt=".1f", cmap="Blues")
plt.title("Entity Group Distribution by Outlet (%) Without Tagesschau")
plt.xlabel("Entity group")
plt.ylabel("Source")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Start from the full framing results df
raw_hit_df = df.copy()

# Keep only non-neutral / non-irrelevant rows and exclude Tagesschau
raw_hit_df = raw_hit_df[
    (~raw_hit_df["category"].isin(["NEUTRAL", "IRRELEVANT"])) &
    (raw_hit_df["source"] != "Tagesschau")
].copy()

# Build a stable context key
if "context_idx" in raw_hit_df.columns:
    raw_hit_df["context_key"] = (
        raw_hit_df["row_id"].astype(str) + "||" + raw_hit_df["context_idx"].astype(str)
    )
else:
    raw_hit_df["context_key"] = (
        raw_hit_df["row_id"].astype(str) + "||" + raw_hit_df["context_window"].fillna("").astype(str)
    )

# Split hit_text into raw hit words
raw_hit_df["raw_hit"] = raw_hit_df["hit_text"].fillna("").astype(str).str.split(r"\s*\|\s*")
raw_hit_df = raw_hit_df.explode("raw_hit").reset_index(drop=True)
raw_hit_df["raw_hit"] = raw_hit_df["raw_hit"].fillna("").astype(str).str.strip()
raw_hit_df = raw_hit_df[raw_hit_df["raw_hit"] != ""].copy()

# Deduplicate: same outlet-context-hit should count once
raw_hit_df = raw_hit_df.drop_duplicates(subset=["source", "context_key", "raw_hit"]).copy()

source_order_no_tagesschau = [
    s for s in [
        "Antispiegel", "Compact", "Deutschlandkurier",
        "Nius", "RT_de", "Tichys_Einblick"
    ]
    if s in raw_hit_df["source"].unique()
]

# Counts for all raw hits
raw_hit_counts = pd.crosstab(
    raw_hit_df["source"],
    raw_hit_df["raw_hit"]
).reindex(
    index=source_order_no_tagesschau,
    fill_value=0
)

# Percentages within each outlet
raw_hit_pct = (
    raw_hit_counts
    .div(raw_hit_counts.sum(axis=1), axis=0)
    .mul(100)
    .round(2)
)

# Keep top raw hits for visualization
TOP_N_HITS = 20
top_hits = raw_hit_counts.sum(axis=0).sort_values(ascending=False).head(TOP_N_HITS).index.tolist()

raw_hit_counts_top = raw_hit_counts[top_hits]
raw_hit_pct_top = raw_hit_pct[top_hits]

display(raw_hit_counts_top)
display(raw_hit_pct_top)

plt.figure(figsize=(14, 6))
sns.heatmap(raw_hit_pct_top, annot=True, fmt=".1f", cmap="Blues")
plt.title(f"Raw Hit Distribution by Outlet (%) Without Tagesschau\nTop {TOP_N_HITS} raw hits")
plt.xlabel("Raw hit")
plt.ylabel("Source")
plt.tight_layout()
plt.show()
